<a href="https://colab.research.google.com/github/tusharj23/ProteinBERT/blob/main/notebooks/ProteinBERT_with_pretraining_ensemble_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/nadavbra/protein_bert.git
%cd protein_bert
!git submodule init
!git submodule update

!pip install git+https://github.com/tensorflow/addons.git
!pip install .
!pip install pandas numpy h5py lxml pyfaidx
!pip install tape-proteins

Cloning into 'protein_bert'...
remote: Enumerating objects: 253, done.
remote: Counting objects: 100% (99/99), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 253 (delta 84), reused 61 (delta 61), pack-reused 154 (from 1)
Receiving objects: 100% (253/253), 23.44 MiB | 30.08 MiB/s, done.
Resolving deltas: 100% (128/128), done.
/content/protein_bert
Submodule 'proteinbert/shared_utils' (https://github.com/nadavbra/shared_utils.git) registered for path 'proteinbert/shared_utils'
Cloning into '/content/protein_bert/proteinbert/shared_utils'...
Submodule path 'proteinbert/shared_utils': checked out 'dc1c62a1754c51f6d46b7486e4a3e5e62c0570e1'
  Cloning https://github.com/tensorflow/addons.git to /tmp/pip-req-build-u8c414_y
  Running command git clone --filter=blob:none --quiet https://github.com/tensorflow/addons.git /tmp/pip-req-build-u8c414_y
  Resolved https://github.com/tensorflow/addons.git to commit d208d752e98c310280938efa939117bf635a60a8
  Installing build depende

In [2]:
!wget https://s3.amazonaws.com/songlabdata/proteindata/data_pytorch/secondary_structure.tar.gz
!mkdir -p data
!tar -xzf secondary_structure.tar.gz -C data

--2026-04-01 11:51:30--  https://s3.amazonaws.com/songlabdata/proteindata/data_pytorch/secondary_structure.tar.gz
Resolving s3.amazonaws.com (s3.amazonaws.com)... 52.217.126.80, 16.15.199.30, 52.217.164.24, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|52.217.126.80|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 251794897 (240M) [application/x-tar]
Saving to: ‘secondary_structure.tar.gz’

secondary_structure 100%[===================>] 240.13M  42.6MB/s    in 6.0s    

2026-04-01 11:51:37 (40.3 MB/s) - ‘secondary_structure.tar.gz’ saved [251794897/251794897]



In [3]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from proteinbert import load_pretrained_model
from proteinbert.conv_and_global_attention_model import get_model_with_hidden_layers_as_outputs
from tape.datasets import SecondaryStructureDataset

In [4]:
from proteinbert.conv_and_global_attention_model import get_model_with_hidden_layers_as_outputs

seq_len = 512

pretrained_model_generator, input_encoder = load_pretrained_model()

model_with_hidden_layers = get_model_with_hidden_layers_as_outputs(
    pretrained_model_generator.create_model(seq_len)
)

model = model_with_hidden_layers
print("ProteinBERT hidden layer model loaded")

 Local model dump file /root/proteinbert_models/default.pkl doesn't exist. Will download https://media.githubusercontent.com/media/Brandes-Lab/proteinbert_data_files/master/epoch_92400_sample_23500000.pkl into /root/proteinbert_models. Please approve or reject this (to exit and potentially call the function again with different parameters).
Do you approve downloading the file into the specified directory? Please specify "Yes" or "No":Yes
Downloaded file: /root/proteinbert_models/epoch_92400_sample_23500000.pkl
Created: /root/proteinbert_models/default.pkl
ProteinBERT hidden layer model loaded


In [5]:
train_dataset = SecondaryStructureDataset(
    data_path="./data",
    split="train"
)

valid_dataset = SecondaryStructureDataset(
    data_path="./data",
    split="valid"
)

test_dataset = SecondaryStructureDataset(
    data_path="./data",
    split="cb513"
)

print(len(train_dataset), len(valid_dataset), len(test_dataset))

8678 2170 513


In [6]:
from torch.utils.data import Subset
import numpy as np

num_models = 8

indices = np.arange(len(train_dataset))
np.random.shuffle(indices)

split_size = len(indices) // num_models

train_splits = []

for i in range(num_models):

    start = i * split_size

    if i == num_models - 1:
        split_indices = indices[start:]
    else:
        split_indices = indices[start:start + split_size]

    train_splits.append(Subset(train_dataset, split_indices))

print([len(split) for split in train_splits])

[1084, 1084, 1084, 1084, 1084, 1084, 1084, 1090]


In [7]:
id_to_aa = {
    0:'PAD', 1:'A', 2:'C', 3:'D', 4:'E', 5:'F',
    6:'G', 7:'H', 8:'I', 9:'K', 10:'L',
    11:'M', 12:'N', 13:'P', 14:'Q', 15:'R',
    16:'S', 17:'T', 18:'V', 19:'W', 20:'Y',
    21:'X', 22:'B', 23:'Z', 24:'J', 25:'U', 26:'O'
}

In [8]:
def tokens_to_sequence(token_ids):
    seq = ""
    for t in token_ids:
        if t == 0:
            continue
        seq += id_to_aa.get(int(t), 'X')
    return seq

In [9]:
def encode_dataset(dataset, max_len=510, num_samples=500):

    sequences = []
    labels = []

    count = 0
    i = 0

    while count < num_samples:
        token_ids, mask, label = dataset[i]
        i += 1

        seq = tokens_to_sequence(token_ids)

        if len(seq) < 5:
            continue

        seq = seq[:max_len]
        label = label[:max_len]

        if len(seq) < max_len:
            seq = seq + "X" * (max_len - len(seq))

        if len(label) < max_len:
            pad_len = max_len - len(label)
            label = np.concatenate([label, np.full(pad_len, -1)])

        sequences.append(seq)
        labels.append(label)

        count += 1

    encoded_x = input_encoder.encode_X(sequences, seq_len=max_len)
    encoded_x = encoded_x[0]
    labels = np.array(labels).astype(np.int32)

    return encoded_x, labels

In [10]:
max_len = seq_len - 2

X_val,y_val = encode_dataset(valid_dataset,max_len,500)
X_test,y_test = encode_dataset(test_dataset,max_len,200)

In [11]:
def extract_flat_embeddings(X, y, batch_size=2):

    X_flat = []
    y_flat = []

    for start in range(0, len(X), batch_size):

        end = start + batch_size
        seq_batch = X[start:end]

        go_batch = np.zeros((len(seq_batch), 8943))

        outputs = model.predict([seq_batch, go_batch], verbose=0)
        local_batch = outputs[0]   # local embeddings

        for b in range(len(seq_batch)):
            labels_seq = y[start + b]

            for j in range(len(labels_seq)):
                if labels_seq[j] != -1:
                    embedding = local_batch[b, j, :]
                    X_flat.append(embedding)
                    y_flat.append(labels_seq[j])

        print("Processed batch:", start)

    return np.array(X_flat), np.array(y_flat)

In [12]:
print("Val embeddings...")
X_val_flat,y_val_flat=extract_flat_embeddings(X_val,y_val)

print("Test embeddings...")
X_test_flat,y_test_flat=extract_flat_embeddings(X_test,y_test)

Val embeddings...
Processed batch: 0
Processed batch: 2
Processed batch: 4
Processed batch: 6
Processed batch: 8
Processed batch: 10
Processed batch: 12
Processed batch: 14
Processed batch: 16
Processed batch: 18
Processed batch: 20
Processed batch: 22
Processed batch: 24
Processed batch: 26
Processed batch: 28
Processed batch: 30
Processed batch: 32
Processed batch: 34
Processed batch: 36
Processed batch: 38
Processed batch: 40
Processed batch: 42
Processed batch: 44
Processed batch: 46
Processed batch: 48
Processed batch: 50
Processed batch: 52
Processed batch: 54
Processed batch: 56
Processed batch: 58
Processed batch: 60
Processed batch: 62
Processed batch: 64
Processed batch: 66
Processed batch: 68
Processed batch: 70
Processed batch: 72
Processed batch: 74
Processed batch: 76
Processed batch: 78
Processed batch: 80
Processed batch: 82
Processed batch: 84
Processed batch: 86
Processed batch: 88
Processed batch: 90
Processed batch: 92
Processed batch: 94
Processed batch: 96
Process

In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

def build_classifier():

    clf = Sequential([

        Dense(256, activation='relu', input_shape=(1562,)),
        Dropout(0.3),

        Dense(128, activation='relu'),

        Dense(3, activation='softmax')

    ])

    clf.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return clf

In [14]:
ensemble_models=[]

for i in range(8):

    print("\nTraining model",i+1)

    X_train,y_train=encode_dataset(

        train_splits[i],
        max_len,
        len(train_splits[i])

    )

    X_train_flat,y_train_flat=extract_flat_embeddings(

        X_train,
        y_train

    )

    clf=build_classifier()

    clf.fit(

        X_train_flat,
        y_train_flat,

        validation_data=(

            X_val_flat,
            y_val_flat

        ),

        epochs=10,
        batch_size=256

    )

    ensemble_models.append(clf)


Training model 1
Processed batch: 0
Processed batch: 2
Processed batch: 4
Processed batch: 6
Processed batch: 8
Processed batch: 10
Processed batch: 12
Processed batch: 14
Processed batch: 16
Processed batch: 18
Processed batch: 20
Processed batch: 22
Processed batch: 24
Processed batch: 26
Processed batch: 28
Processed batch: 30
Processed batch: 32
Processed batch: 34
Processed batch: 36
Processed batch: 38
Processed batch: 40
Processed batch: 42
Processed batch: 44
Processed batch: 46
Processed batch: 48
Processed batch: 50
Processed batch: 52
Processed batch: 54
Processed batch: 56
Processed batch: 58
Processed batch: 60
Processed batch: 62
Processed batch: 64
Processed batch: 66
Processed batch: 68
Processed batch: 70
Processed batch: 72
Processed batch: 74
Processed batch: 76
Processed batch: 78
Processed batch: 80
Processed batch: 82
Processed batch: 84
Processed batch: 86
Processed batch: 88
Processed batch: 90
Processed batch: 92
Processed batch: 94
Processed batch: 96
Process

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1040/1040 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - accuracy: 0.5675 - loss: 0.9116 - val_accuracy: 0.5848 - val_loss: 0.8914
Epoch 2/10
1040/1040 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.5917 - loss: 0.8763 - val_accuracy: 0.5931 - val_loss: 0.8751
Epoch 3/10
1040/1040 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.5999 - loss: 0.8640 - val_accuracy: 0.6034 - val_loss: 0.8626
Epoch 4/10
1040/1040 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.6047 - loss: 0.8552 - val_accuracy: 0.6027 - val_loss: 0.8657
Epoch 5/10
1040/1040 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6093 - loss: 0.8474 - val_accuracy: 0.6031 - val_loss: 0.8663
Epoch 6/10
1040/1040 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.6137 - loss: 0.8404 - val_accuracy: 0.6011 - val_loss: 0.8627
Epoch 7/10
1040/1040 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.6171 - loss: 0.8339 - val_accuracy: 0.6016 - val_loss: 0.8640
Epoch 8/10
1040/1040 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.6222 - loss: 0.8272 -

In [15]:
def ensemble_predict(models,X):

    preds=[]

    for model in models:

        preds.append(model.predict(X))

    avg_preds=np.mean(preds,axis=0)

    return np.argmax(avg_preds,axis=1)

In [16]:
from sklearn.metrics import accuracy_score

y_pred=ensemble_predict(

    ensemble_models,

    X_test_flat

)

acc=accuracy_score(

    y_test_flat,

    y_pred

)

print("Final Ensemble Accuracy:",acc)

1672/1672 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
1672/1672 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
1672/1672 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
1672/1672 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
1672/1672 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
1672/1672 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
1672/1672 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step
1672/1672 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
Final Ensemble Accuracy: 0.609518466573165
